# Python packages installation

A good [introduction](https://github.com/astrofrog/py4sci) to Python in [Jupyter Notebooks](http://jupyter.kip.uni-heidelberg.de/) is recommended for students that are new to using Python for data acquisition and analysis.

We are using the [JupyterLab](https://jupyter.org/) server with the [SciPy Notebook](https://hub.docker.com/r/jupyter/scipy-notebook) that is available for free and can be used on any computer.

First we install all additional Python packages required for this experiment with pip.

In [1]:
!pip3 install pyvisa-py pyserial pyusb zeroconf

  Using cached PyVISA_py-0.7.0-py3-none-any.whl (70 kB)
  Using cached pyserial-3.5-py2.py3-none-any.whl (90 kB)
  Using cached pyusb-1.2.1-py3-none-any.whl (58 kB)
  Obtaining dependency information for zeroconf from https://files.pythonhosted.org/packages/0c/15/259cad0772f17ec0f5bdb48320a5447dfc319ab019c6ba7657d20eccb282/zeroconf-0.119.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata
  Using cached zeroconf-0.119.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.0 kB)
  Using cached PyVISA-1.13.0-py3-none-any.whl (175 kB)
  Using cached ifaddr-0.2.0-py3-none-any.whl (12 kB)
Using cached zeroconf-0.119.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (7.2 MB)


# Communication with hardware


**Important:** All devices have to be enabled _prior_ to starting the JupyterLab server. If communication with hardware fails, shut down the server, enable _all_ devices, then start the JupyterLab server again.

## **PyVISA** library

**V**irtual **I**nstrument **S**oftware **A**rchitecture ([VISA](https://en.wikipedia.org/wiki/Virtual_instrument_software_architecture)) is a widely used application programming interface (API) in the test and measurement (T&M) industry for communicating with instruments from a computer.

* [PyVISA](https://pyvisa.readthedocs.io/en/latest/) is a Python package that allows to control all kinds of measurement devices independently of the interface (e.g. GPIB, RS232, USB, Ethernet).

* [PyVISA-py](https://pyvisa.readthedocs.io/projects/pyvisa-py/en/latest/) is a backend for **PyVISA**. It implements most of the methods for Message Based communication (Serial, USB, GPIB, Ethernet) using Python and some well developed, easy to deploy and cross platform libraries.

After importing the **PyVISA** library via ``import pyvisa`` the **PyVISA-py** backend can be used by choosing ``@py`` when instantiating the visa Resource Manager. 

Invoking the ``list_resources()`` method provides all locally available resources. 

## **USB** adapters for communication

The adapters have to be attached to the computer _and_ the devices enabled _before_ starting the JupyterLab Server, because the kernel module ``ftdi_sio`` will be reloaded with ``modprobe -r ftdi_sio ; modprobe ftdi_sio`` on starting the server and then probe the devices and configure their communication settings according to the devices attached. If communication does not work (due to devices not attached or enabled prior to starting the server) restarting the JupyterLab Server (closing the terminal and starting it again) may fix the issue.

In Linux the devices with USB ID ``0403:6001`` will be used by the kernel driver ``ftdi_sio`` that creates device files at ``/dev/ttyUSB0`` ...  ``/dev/ttyUSB1`` etc. with the lowest free name available. 

When there are multiple devices, the udev system allows to distinguish (e.g. via attributes in ``udevadm info --attribute-walk --name=ttyUSB0``) and create a stable mapping with udev rules in ``/etc/udev/rules.d/`` to ensure that unique symlinks always point to the same devices, even in case they happen to end up with other device files:

<!---
 * The [US232R-100-BULK](https://ftdichip.com/wp-content/uploads/2023/07/DS_US232R-10_R-100-500.pdf) is a generic USB to serial (RS232) adapter manufactured by [FTDI](https://ftdichip.com/). The file ``99-ftdi.rules`` creates ``/dev/ttyUSBSerial`` with ``SUBSYSTEMS=="usb", ATTRS{idVendor}=="0403", ATTRS{idProduct}=="6001", ATTRS{manufacturer}=="FTDI", ATTRS{product}=="US232R", MODE="0666", SYMLINK+="ttyUSBSerial"`` which will be listed as ``ASRL/dev/ttyUSBSerial::INSTR`` by **PyVISA**. 
-->

* The [Prologix GPIB-USB controller](https://prologix.biz/downloads/PrologixGpibUsbManual-6.0.pdf) is a USB bridge to GPIB (IEEE-488) adapter manufactured by [Prologix ](https://prologix.biz/). The file ``99-prologix.rules`` creates ``/dev/ttyUSBGPIB`` with ``SUBSYSTEMS=="usb", ATTRS{idVendor}=="0403", ATTRS{idProduct}=="6001", ATTRS{manufacturer}=="Prologix", ATTRS{product}=="Prologix GPIB-USB Controller", MODE="0666", SYMLINK+="ttyUSBGPIB"`` which will be listed as ``ASRL/dev/ttyUSBSerial::INSTR`` by **PyVISA**.

In this setup we use the Prologix GPIB-USB controller as an USB bridge to GPIB devices such as the Synthesized Function Generator (GPIB Address 19) or the Lock-In Amplifier (GPIB Address 7).

<!--
In addition we can use the generic FTDI USB to serial adapter to communicate with RS232 ports, in this case we need to ensure correct crossover mapping between receive (RXD) and transmit (TXD) pins of the RS232 cable (RXD@adapter pin 2 connected with TXD@device pin 3) and (TXD@adapter pin 3 connected with RXD@device pin 2), e.g. a "cross-over" cable with two female 9-pin RS232 connectors.
-->


In [2]:
import pyvisa # hardware access

rm = pyvisa.ResourceManager('@py') # choosing the PyVISA-py backend
print(rm.list_resources())         # list local hardware resources

('ASRL/dev/ttyUSBGPIB::INSTR',)


## **Prologix GPIB-USB** controller

The [Prologix GPIB-USB controller](https://prologix.biz/downloads/PrologixGpibUsbManual-6.0.pdf) converts any computer with a USB port into a **G**eneral **P**urpose **I**nterface **B**us ([GPIB](https://en.wikipedia.org/wiki/IEEE-488)) Controller or Device.

We use the Prologix GPIB-USB controller as an USB bridge to GPIB, it apppears as [FTDI](https://ftdichip.com/) FT232 (UART IC USB bridge with USB ID ``0403:6001``) as serial device ``/dev/ttyUSBGPIB`` in Linux and can be accessed as ``ASRL/dev/ttyUSBGPIB::INSTR`` in **PyVISA**.

Querying the resource with ``++ver`` asks the controller for its version, it will respond with its firmware version string.

In [3]:
import pyvisa # hardware access

rm  = pyvisa.ResourceManager('@py')                  # choosing the PyVISA-py backend
res = rm.open_resource('ASRL/dev/ttyUSBGPIB::INSTR') # open Prologix GPIB-USB controller
print(res.query("++ver"))                            # Query Prologix GPIB-USB controller for its firmware version

Prologix GPIB-USB Controller version 6.107



The Prologix GPIB-USB controller interprets high level commands received from the host computer and performs the appropriate low-level GPIB protocol
handshaking.

* In **Device mode** (``++mode 0``), Prologix GPIB-USB controller converts the computer into a GPIB peripheral for downloading data and screen plots from the instrument front panel. 

* In **Controller mode** (``++mode 1``), Prologix GPIB-USB controller can remotely control GPIB enabled instruments such as Oscilloscopes, Logic Analyzers, and Spectrum Analyzers.

In our case we will use the Prologix GPIB-USB controller in controller mode with ``++mode 1`` and choose to talk to the GPIB device at address 19 via ``++addr 19``.

All commands to the controller have a prefix of ``++`` while the everything else will be written on GPIB to the device address. 

The chars ESC (``x1b``), CR (``x0d``), LF (``x0a``) and PLUS (``x1b``) have to be escaped by an ESC (``x1b``) prefix.

In [4]:
import pyvisa # hardware access

rm  = pyvisa.ResourceManager('@py')                  # Choose the PyVISA-py backend
res = rm.open_resource('ASRL/dev/ttyUSBGPIB::INSTR') # Open Prologix GPIB-USB controller
res.write("++mode 1")                                # Put Prologix GPIB-USB in Controller-In-Charge (CIC) mode on the GPIB
res.write("++addr 19")                               # Choose device on GPIB address 19 for further communication
res.write("FREQ 123.456")                            # Write a command to this device on GPIB

14

## **Python classes**

We will use Python [classes](https://docs.python.org/3/tutorial/classes.html) to implement functionality in a hierarchy of sections that naturally divide a task into smaller sub-tasks that can be independently implemented and verified. 

Once a sub-task is finished the next sub-task can inherit the methods and attributes of the previous sub-task and this hierarchy can be expressed in individual Python classes.

The first argument in each method of a Python class is ```self``` referring to this instance of the Python class and can be used to to address attributes and methods of this instance.

In [5]:
class Base():                                                    # Class name
    def __init__(self, name="Default", purpose="communication"): # Constructor gets called when instance of class is created
        self.name = name                                         # Attribute of instance "self" called "name" will be set to value of variable "name"
        self.purpose = purpose                                   # Attribute set
        self._setText("Class {name} for {purpose}.")             # Call method on this instance "self"

    def _setText(self, text):                                    # Method definition, first argument always is "self" refering to this instance
        self._text = text                                        # Attribute set, underscore prefix indicates internal usage

    def help(self):                                              # Method definition, first argument always is "self" refering to this instance
        print(self._text.format(name=self.name, purpose=self.purpose))


base = Base("GPIB", "hardware communication")                    # Instance of class created and constructor called with arguments. Inside the class the self reference is used for this
base.help()        

Class GPIB for hardware communication.


## **GPIB** class for Prologix GPIB-USB controller

A generic Python class for GPIB devices via the [Prologix GPIB-USB controller](https://prologix.biz/downloads/PrologixGpibUsbManual-6.0.pdf).

In [6]:
import pyvisa, re

class GPIB(Base): # Create class GPIB that interits the methods and attributes of the Base class
    """Use GPIB devices via Prologix GPIB-USB Controller Rev 6.4.1"""
    # https://pyvisa.readthedocs.io/en/latest/
    # https://prologix.biz/downloads/PrologixGpibUsbManual-6.0.pdf

    def __init__(self, dev="ttyUSBGPIB", addr=None, res=None):   # Constructor
        self.__parent = super(GPIB, self)                        # https://docs.python.org/3/library/functions.html#super
        self.__parent.__init__("GPIB", "hardware communication") # call constructor of parent class with arguments      
        self._rm   = pyvisa.ResourceManager("@py")               # using PyVISA-py
        self._dev  = self.setDev(dev)   if dev  else None
        self._addr = self.setAddr(addr) if addr else None
        self._res  = self.setRes(res)   if res  else self._res
        self._esc  = ['\x1b', '\x0d', '\x0a', '\x2b']

    def setDev(self, dev=None):
        if dev:
            self._dev = str(dev)
        if self._rm and self._dev:
            for res in sorted(self._rm.list_resources()):
                if self._dev in res:
                    self._res = self._rm.open_resource(res)
                    self.setRes(self._res)
        return self._dev
    
    def setRes(self, res=None):
        if res:
            self._res = res
        if self._res:
            self._res.read_termination  = "\n"
            self._res.write_termination = "\n"
            self._res.write("++mode 1") # "++mode" is "0" (GPIB TALKER or GPIB LISTENER) or "1" (Controller-In-Charge (CIC))
        return self._res        

    def setAddr(self, addr=None):
        if addr:
            self._addr = int(addr)
        if self._res and self._addr:
            self._res.write("++addr " + str(self._addr))
        return self._addr
                
    def _escape(self, data):
        # xlat_t = { ord(c):'\x1b' + c for c in '\x1b\r\n+' }; '+A\rB\nC\x1b'.translate(xlat_t) #==> '\x1b+A\x1b\rB\x1b\nC\x1b\x1b'
        for b in ['\x1b', '\x0d', '\x0a', '\x2b']: # ESC, CR, LF, PLUS (ESC MUST BE FIRST)
            data = data.replace(b, '\x1b' + b) # prepend with ESC
        return data
            
    def write(self, addr, data):
        if addr and data:
            self.setAddr(addr)
            if self._res:
                return self._res.write(self._escape(data))

    def query(self, addr, data, dtype="float"):
        if addr and data and dtype:
            self.setAddr(addr)
            if self._res:
                if dtype == "float":
                    return float(re.sub('[^0-9eE\.+-]', '',
                        self._res.query(self._escape(data))
                    ))
                else:
                    return self._res.query(data)


### Below we demonstrate an example usage of an instance from this GPIB() class:

### Prologix GPIB-USB controller
gpib = GPIB("ttyUSBGPIB") # create an instance from the GPIB() class on the device /dev/ttyUSBGPIB
print(gpib._res.query("++ver"))

### call our help method from the parent class
gpib.help()

### SRS SD345 30 MHz Synthesized Function Generator on GPIB address 19
freq = 1234.567 # Frequency in Hz <= 30E6
gpib.write(19, "FREQ {:.3f}".format(freq))
freq = gpib.query(19, "FREQ?")
print("freq = {:.3f} Hz".format(freq))

ampl = 10.00 # Amplitude in Vpp <= 10
gpib.write(19, "AMPL {0:.2f}VP".format(ampl))
ampl = gpib.query(19, "AMPL?")
print("ampl = {0:.2f} Vpp".format(ampl))

### SRS SR530 Lock-In Amplifier on GPIB address 7
qx = gpib.query(7, "QX") # X in V
qy = gpib.query(7, "QY") # Y in V
print("(x,y) = ({x:.2E} V, {y:.2E} V)". format(x=qx, y=qy))

inp_key = "X1" # or "X2" according to hw setup label on lock-in amp
inp_val = gpib.query(7, inp_key) # input in Vdc
print("input = {0:.3f} V". format(inp_val))

out_key = "X5" # or "X6" according to hw setup label on lock-in amp
out_val = 0.512 # output in V e.g. -0.123 or 0.456
gpib.write(7, "{key},{val:.3f}".format(key=out_key, val=out_val))
out_val = gpib.query(7, out_key)
print("output = {0:.3f} V". format(out_val))
gpib.write(7, "{0},0.000".format(out_key)) # reset to 0 (to avoid cooling or heating via peltier element)
print("output reset back to " + str(gpib.query(7, out_key)) + " V")

Prologix GPIB-USB Controller version 6.107
Class GPIB for hardware communication.
freq = 1234.567 Hz
ampl = 10.00 Vpp
(x,y) = (0.00E+00 V, 2.50E-07 V)
input = 0.003 V
output = 0.513 V
output reset back to 0.0 V


## **SynFunGen** class for 30 MHz Synthesized Function Generator SRS DS345

A Python class for communication via GPIB with the 30 MHz Synthesized Function Generator (Stanford Research Systems Model [DS345](https://www.thinksrs.com/products/ds345.html)) inherits from the GPIB class.

In [7]:
# TODO for students: Implement a class SynFunGen that inherits from GPIB class and controls the function generator


## **LockInAmp** class for Lock-In Amplifier SRS SR530

A Python class for communication via GPIB with the Lock-In Amplifier (Stanford Research Systems [SR530](https://www.thinksrs.com/products/sr510530.htm)) that inherits everything from the SynFunGen class (including its parent GPIB class, hence everything for implementation aswell).

In [8]:
# TODO for students: Implement a class LockInAmp that inherits from the function generator class and controls the Lock-In amplifier


## **DataWriter** class for writing data to files

A Python class for data storage into files. For numerical data using TSV or CSV format is suggested because of its widespread usage and software support.

We recommend the [Python CSV](https://docs.python.org/3/library/csv.html) library that provides a [DictWriter()](https://docs.python.org/3/library/csv.html#csv.DictWriter) method for data storage in CSV format.

In [9]:
# TODO for students: Implement a class DataWriter that inherits from the Lock-in amp class and
# provides methods to open a file name, write CSV headers and for each measurement a row of columns.


## **Measurement** class for the measurement sequence

Now that we have every functionality ready (and individually tested!) we can finally implement the class that automates the measurement tasks according to the lab course manual.

Using [numpy](https://numpy.org/doc/stable/reference/index.html#reference) we recommend the methods [std()](https://numpy.org/doc/stable/reference/generated/numpy.std.html) and [mean()](https://numpy.org/doc/stable/reference/generated/numpy.mean.html) for calculating the mean value and standard deviation (repeated measurements under the same conditions).

For calculating the phase, we recommend the method [arctan2()](https://numpy.org/doc/stable/reference/generated/numpy.arctan2.html#numpy-arctan2) (described [here](https://en.wikipedia.org/wiki/Atan2) in detail).

In [10]:
# TODO for students: Implement a class Measurement that inherits from the data writer class (and therefore everything else)
# for fully autonomous measurement sequences and data storage into individual files.


## **DataReader** class for reading data from files

A Python class for reading data from files. This class should implement reading from the data format we chose for holding our measurement data, hence the TSV or CSV format.

We recommend the [Python CSV](https://docs.python.org/3/library/csv.html) library that provides a [DictReader()](https://docs.python.org/3/library/csv.html#csv.DictReader) method for reading data from CSV files into Pyhton dictionaries.

For each value that is read as string from CSV a conversion to float can be done using the Python [float()](https://docs.python.org/3/library/functions.html#float) method with [exception](https://docs.python.org/3/tutorial/errors.html#handling-exceptions) handling.

In [11]:
# TODO for students: Implement a class DataReader that does not need any inheritance from another class
# for reading back the measurement data into numerical (float) format.


## **DataPlot** class for plotting measurement data

A Python class for plotting measurement data. This class should implement a plotting routine suitable to display the measurement data.

We recommend combining [NumPy](https://numpy.org/) with the [MatPlotLib](https://matplotlib.org/) library that provides [plot()](https://matplotlib.org/stable/plot_types/basic/plot.html#sphx-glr-plot-types-basic-plot-py) and [errorbar()](https://matplotlib.org/stable/plot_types/stats/errorbar_plot.html#sphx-glr-plot-types-stats-errorbar-plot-py) functions.

For creating interactive plots that allow reading out values and zooming via mouse we suggest using the [ipympl](https://matplotlib.org/ipympl/) extension via ```%matplotlib ipympl``` in the cell prefix.

In [12]:
# TODO for students: Implement a class DataPlot that plots the measurement data in interactive plots.


## **DataFit** class for fitting the measurement data

A Python class for fitting the measurement data. This class should implement an appropriate fit function that reflects the physics of the measurement data and allows extracting fit parameters.

We recommend using the method [curve_fit()](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html) from the [SciPy](https://scipy.org/) library.

In [13]:
# TODO for students: Implement a class DataFit that fits the measurement data in interactive plots.
